In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
my_schema = StructType([
    StructField("row_id", IntegerType(), True),
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("ship_date", StringType(), True),
    StructField("ship_mode", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("region", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("sales_clean", DoubleType(), True),

])

In [0]:
df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(my_schema) \
    .load("/Volumes/pyspark_cata/default/default_volume/source/")

In [0]:
def clean_supply_chain_data(df):

    # 1. Handle null values
    df = df.fillna({
        "sales_clean": 0
    })

    # 2. Remove duplicates (business key based)
    df = df.dropDuplicates(["order_id","product_id"])

    return df

In [0]:
df = clean_supply_chain_data(df)

In [0]:
from pyspark.sql.functions import to_date, col
def convert_to_date(df):
    df = df.withColumn(
    "order_date",
    expr("try_to_date(order_date, 'dd/MM/yyyy')")
    ).withColumn(
    "ship_date",
    expr("try_to_date(ship_date, 'dd/MM/yyyy')")
    )

    df = df.withColumn(
    "order_date",
    expr("""
        coalesce(
            try_to_date(order_date, 'dd-MM-yyyy'),
            try_to_date(order_date, 'dd/MM/yyyy')
        )
    """)
    ).withColumn(
    "ship_date",
    expr("""
        coalesce(
            try_to_date(ship_date, 'dd-MM-yyyy'),
            try_to_date(ship_date, 'dd/MM/yyyy')
        )
    """)
    )


    return df

In [0]:
df = convert_to_date(df)

In [0]:
df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "/Volumes/pyspark_cata/default/default_volume/checkpoints_2/checkpoint_file_2"
    ) \
    .trigger(availableNow=True) \
    .toTable("pyspark_cata.default.sink_2")

In [0]:
%sql
select * from pyspark_cata.default.sink_2

*Dim Products*

In [0]:
%sql
DROP TABLE IF EXISTS pyspark_cata.default.dim_product;

CREATE TABLE pyspark_cata.default.dim_product
USING DELTA
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY product_id) AS dim_product_key,
    product_id,
    category,
    sub_category
FROM (
    SELECT DISTINCT product_id, category, sub_category
    FROM pyspark_cata.default.sink_2
);


In [0]:
%sql

-- DROP TABLE IF EXISTS pyspark_cata.default.dim_product;

-- CREATE TABLE pyspark_cata.default.dim_product (
--     dim_product_key BIGINT,
--     product_id STRING,
--     product_name STRING,
--     category STRING,
--     sub_category STRING
-- )
-- USING DELTA;


In [0]:
%sql
-- MERGE INTO pyspark_cata.default.dim_product d
-- USING (
--     SELECT DISTINCT
--         product_id,
--         product_name,
--         category,
--         sub_category
--     FROM pyspark_cata.default.sink_2
-- ) s
-- ON d.product_id = s.product_id

-- WHEN MATCHED THEN
-- UPDATE SET
--     d.product_name = s.product_name,
--     d.category     = s.category,
--     d.sub_category = s.sub_category

-- WHEN NOT MATCHED THEN
-- INSERT (
--     dim_product_key,
--     product_id,
--     product_name,
--     category,
--     sub_category
-- )
-- VALUES (
--     (SELECT COALESCE(MAX(dim_product_key), 0) + 1 FROM pyspark_cata.default.dim_product),
--     s.product_id,
--     s.product_name,
--     s.category,
--     s.sub_category
-- );


*Dim_Location*

In [0]:
%sql
DROP TABLE IF EXISTS pyspark_cata.default.dim_location;

CREATE TABLE pyspark_cata.default.dim_location
USING DELTA
AS
SELECT
    postal_code,
    city,
    state,
    region,
    country,
    ROW_NUMBER() OVER (
        ORDER BY postal_code, city, state
    ) AS dim_location_key
FROM (
    SELECT DISTINCT postal_code, city, state, region, country
    FROM pyspark_cata.default.sink_2
);


In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS pyspark_cata.default.dim_location (
--     dim_location_key BIGINT,
--     postal_code STRING,
--     city STRING,
--     state STRING,
--     region STRING,
--     country STRING
-- )
-- USING DELTA;


In [0]:
%sql
-- MERGE INTO pyspark_cata.default.dim_location d
-- USING (
--     SELECT DISTINCT
--         postal_code,
--         city,
--         state,
--         region,
--         country
--     FROM pyspark_cata.default.sink_2
-- ) s
-- ON  d.postal_code = s.postal_code
-- AND d.city        = s.city
-- AND d.state       = s.state

-- WHEN MATCHED THEN
-- UPDATE SET
--     d.region  = s.region,
--     d.country = s.country

-- WHEN NOT MATCHED THEN
-- INSERT (
--     dim_location_key,
--     postal_code,
--     city,
--     state,
--     region,
--     country
-- )
-- VALUES (
--     (SELECT COALESCE(MAX(dim_location_key), 0) + 1 FROM pyspark_cata.default.dim_location),
--     s.postal_code,
--     s.city,
--     s.state,
--     s.region,
--     s.country
-- );


*Dim_Customers*

In [0]:
%sql
DROP TABLE IF EXISTS pyspark_cata.default.dim_customer;

CREATE TABLE pyspark_cata.default.dim_customer
USING DELTA
AS
SELECT
    customer_id,
    customer_name,
    segment,
    ROW_NUMBER() OVER (
        ORDER BY customer_id
    ) AS dim_customer_key
FROM (
    SELECT DISTINCT customer_id, customer_name, segment
    FROM pyspark_cata.default.sink_2
);


In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS pyspark_cata.default.dim_customer (
--     dim_customer_key BIGINT,
--     customer_id STRING,
--     customer_name STRING,
--     segment STRING
-- )
-- USING DELTA;


In [0]:
%sql
-- MERGE INTO pyspark_cata.default.dim_customer d
-- USING (
--     SELECT DISTINCT
--         customer_id,
--         customer_name,
--         segment
--     FROM pyspark_cata.default.sink_2
-- ) s
-- ON d.customer_id = s.customer_id

-- WHEN MATCHED THEN
-- UPDATE SET
--     d.customer_name = s.customer_name,
--     d.segment       = s.segment

-- WHEN NOT MATCHED THEN
-- INSERT (
--     dim_customer_key,
--     customer_id,
--     customer_name,
--     segment
-- )
-- VALUES (
--     (SELECT COALESCE(MAX(dim_customer_key), 0) + 1 FROM pyspark_cata.default.dim_customer),
--     s.customer_id,
--     s.customer_name,
--     s.segment
-- );


*Fact_Sales*

In [0]:
%sql
CREATE OR REPLACE TABLE pyspark_cata.default.fact_sales
USING DELTA
AS
SELECT
    s.row_id,
    s.order_id,
    s.order_date,
    s.ship_date,
    s.product_id,
    s.sales_clean AS sales,
    c.dim_customer_key,
    p.dim_product_key,
    l.dim_location_key
FROM pyspark_cata.default.sink_2 s
LEFT JOIN pyspark_cata.default.dim_customer c
    ON s.customer_id = c.customer_id
LEFT JOIN pyspark_cata.default.dim_product p
    ON s.product_id = p.product_id
LEFT JOIN pyspark_cata.default.dim_location l
    ON  s.postal_code = l.postal_code
    AND s.city        = l.city
    AND s.state       = l.state
    ORDER BY row_id;

In [0]:
%sql
select * from pyspark_cata.default.fact_sales